In [ ]:
# Install required dependencies
print("Installing required dependencies...")

!sudo apt-get install -y libusb-1.0-0-dev

# Clone the MuJoCo repo for ALOHA
!git clone https://github.com/google-deepmind/mujoco_menagerie.git

# Clone the Wiki-GRx-Models repo for GR1 and copy the urdf file to temp folder
!git clone https://github.com/FFTAI/Wiki-GRx-Models.git
!cp -r Wiki-GRx-Models/GRX/GR1/gr1t2 gr1_assets

# !pip install --ignore-installed blinker
!pip install -q "z3-solver==4.15.4.0" "genesis-world==0.3.11" "rerun-sdk[all]==0.28.2" "imageio[ffmpeg]"
!pip uninstall -y numpy numba flash-attn
!pip install -q "numpy==2.2.0" "numba>=0.61.2"
!pip install -q "transformers<5.0.0" "lerobot[intelrealsense,dynamixel]==0.4.3"
!pip install -q wandb
!pip install -U flash-attn --no-build-isolation

print("Dependencies installed, restarting!")
exit()

In [ ]:
"""
Flash Attention works for A100 and above, disable for lower spec GPUs
"""

# import os

# patch_code = """
# from transformers import PreTrainedModel
# _orig_check = PreTrainedModel._check_and_adjust_attn_implementation

# def patched_check_attn(self, attn_implementation, is_init_check):
#     if attn_implementation == 'flash_attention_2':
#         return 'sdpa'
#     return _orig_check(self, attn_implementation, is_init_check)

# PreTrainedModel._check_and_adjust_attn_implementation = patched_check_attn
# """

# # The traceback shows lerobot-train points here:
# target_file = "/usr/local/lib/python3.12/dist-packages/lerobot/scripts/lerobot_train.py"

# with open(target_file, "r") as f:
#     content = f.read()

# # Only patch if we haven't already
# if "patched_check_attn" not in content:
#     with open(target_file, "w") as f:
#         f.write(patch_code + "\n" + content)
#     print("Successfully patched lerobot_train.py to disable Flash Attention!")
# else:
#     print("Script was already patched.")

## Pickup Data Cup

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_cup
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_cup
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_cup

In [ ]:
import wandb
wandb.login()

In [ ]:
!hf auth login

In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'IDENTITY'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_cup \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=... \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-cup \
    --wandb.disable_artifact=true

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

## Pickup Data Grasp

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_grasp
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_grasp
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_grasp

In [ ]:
import wandb
wandb.login()

In [ ]:
!hf auth login

In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=new_embodiment \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'IDENTITY'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_grasp \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=... \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-grasp \
    --wandb.disable_artifact=true

In [ ]:
from google.colab import drive
drive.flush_and_unmount()